# Validating Sophisticated Inference (SI) Planning Algorithm using the T-Maze Task

### Overview

This notebook provides a comparative test of two planning algorithms within the active inference framework:
- Vanilla Active Inference Planning: Standard and default planning approach in `pymdp` that evaluates policies and calculates expected free energy calculations, where the agent considers "what would happen if I did that".
- Sophisticated Inference (SI) Planning: A more complex but thorough planning approach that evaluates policies using recursive expected free energy calculations, where the agent considers "what would I believe about what would happen if I did that".

We employ the T-Maze task as described in the [sophisticated inference paper](https://discovery.ucl.ac.uk.id/eprint/10124606/) to validate the SI implementation. 

### The T-Maze Task

The T-maze is a classic sequential decision-making problem where the agent (analogised to a rat) faces a fundamental exploration vs. exploitation dilemma. The agent begins at the centre of a T-shaped maze, where there is also a left arm, right arm, and cue location (bottom arm). One arm contains reward (cheese), the other punishment (shock), but the agent does not know which arm contains what. A cue at the bottom provides information about reward location with a set accuracy.

The agent must choose between immediate but risky exploitation by directly committing to one of the reward arms (50% chance of success) or gather information (explore) first by visiting the cue location at the expense of time, then make an informed choice. When cue validity > 50%, the optimal strategy is to first gather information at the cue location, then select the indicated rewarding arm. This notebook tests whether both planning algorithms can discover this optimal policy and how.

### Notebook Structure
1. Environment Setup: Set up the generative process for the T-Maze environment
2. Agent Setup: Set up the generative model for the agent
3. Active Inference Rollout: Run active inference rollouts with vanilla and sophisticated inference planning algorithms, with optional visualisations of the agent behaviour
4. Results Analysis: Compare actions selected and policy evaluations  

In [1]:
%load_ext autoreload
%autoreload 2

import jax.numpy as jnp
import jax.random as jr

import matplotlib.pyplot as plt
import numpy as np
import mediapy

from pymdp.envs import si_TMaze, rollout
from pymdp.agent import Agent
from pymdp.planning.si import si_policy_search

### Setting up the T-Maze environment (Generative Process)

- The `reward_condition` parameter determines the reward location: `0` for the left arm, `1` for the right arm, or `None` for random allocation.
- The `cue_validity` parameter (default 0.95) represents the accuracy of the cues as a probability.
- The `reward_probability` parameter sets the probability `a` of receiving a reward in the correct arm. 
- With `dependent_outcomes=True` (default), the probability of punishment is `1-a`, otherwise the probability of no-outcome is `1-a`. With `dependent_outcomes=False`, the `punishment_probability` parameter needs to be specified, setting the likelihood of punishment in the other arm.

<details>
<summary> Click here to see how the generative process is set up. </summary>

#### States and Observations

**State Factors:**
1. Location (4 states): 
    - 0: centre (start location)
    - 1: left arm
    - 2: right arm
    - 3: cue location (bottom arm)
2. Reward Location (2 states):
    - 0: reward in left arm 
    - 1: reward in right arm

**Control State Factors:**
1. Location (4 actions): 
    - 0: move to centre (start location)
    - 1: move to left arm
    - 2: move to right arm
    - 3: move to cue location (bottom arm)

**Observation Modalities:**
1. Location (5 observations):
    - 0: centre (start location)
    - 1: left arm
    - 2: right arm
    - 3: cued left arm (bottom arm)
    - 4: cued right arm (bottom arm)
2. Outcome (3 observations):
    - 0: no outcome
    - 1: reward (cheese)
    - 2: punishment (shock)

#### Environment Parameters

**Observation Likelihood Model (A):**
- A[0]: Location observations (5x4x2 tensor)
  - Perfect mapping between true and observed locations.
  - At the cue location (bottom), the rewarding arm is cued with accuracy set by the `cue_validity` parameter.
- A[1]: Outcome observations (3x4x2 tensor)
  - In the rewarding arm (set by `reward_condition`), reward is presented with a likelihood determined by the `reward_probability` parameter.
  - Punishment and/or no-outcome are presented with a likelihood determined depending on if `dependent_outcome` is True or False and consequently by the `punishment_probability` parameter.
  - No-outcome is observed in the centre/start location and cue location.

**Transition Model (B):**
- B[0]: Location transitions (4x4x4 tensor)
  - Agent can move to any one of the four locations from any location regardless of adjacency.
- B[1]: Reward location (2x2x1 tensor)
  - Reward location remains fixed throughout trial.

**Initial Conditions (D):**
- D[0]: Starting location (4x1 tensor)
  - Agent always begins in centre location
- D[1]: Reward placement (2x1 tensor)
  - Default: Equal chance (50/50) of reward in either arm (`reward_condition=None`)
  - Optional: Can fix reward to specific arm, by setting `reward_condition` to `0` (for left arm) or `1` (for right arm)

</details>



In [2]:
# setting the parameters for the environment
reward_condition = 0 # 0 is reward in left arm, 1 is reward in right arm, None is random allocation
cue_validity = 0.95 # 95% valid cues

reward_probability = 1.0 # 100% chance of reward in the correct arm
dependent_outcomes = True # if True, punishment occurs as a function of reward probability (i.e., if reward probability is 0.8, then 20% punishment). If False, punishment occurs with set probability (i.e., 20% no outcome and punishment will only occur in the other (non-rewarding) arm depending on the punishment_probability parameter)

In [3]:
# initialising the environment. see si_tmaze.py in pymdp/envs for the implementation details
env = si_TMaze(
    reward_condition=reward_condition,
    cue_validity=cue_validity,  
    reward_probability=reward_probability,     
    dependent_outcomes=dependent_outcomes,
)

### Setting up the Agents

In [4]:
# creating C tensors filled with zeros for [location], [reward], [cue] based on A shapes for the Agent
C = [jnp.zeros(a.shape[0], dtype=jnp.float32) for a in env.A] 

# setting preferences for outcomes only
C[1] = C[1].at[1].set(2.0)    # prefer reward
C[1] = C[1].at[2].set(-6.0)   # avoid punishment

# slight cost of observing a cue
# C[0] = C[0].at[3].set(-1.0).at[4].set(-1.0)

In [5]:
# flat D tensors [location], [reward] based on B shapes for the agent
D = [jnp.ones(b.shape[0], dtype=jnp.float32) / b.shape[0] for b in env.B]

In [6]:
# note that we initialise agents with different policy lengths for the vanilla vs sophisticated inference planning algorithms
# even though both will eventually end up planning with a horizon of 2. The sophisticated inference planning algorithm requires
# a policy length of 1 in the Agent as we specify horizon length of 2 when initialising the planning algorithm in the `rollout`.

# action_selection="deterministic" means selecting an action from the policy probability distribution (q_pi) by arg-maxxing
# sampling_mode="full" means evaluating the whole action sequence in each policy and executing the first action (as opposed to marginal where the agent evaluates each action type)

agent_vanilla = Agent(
    env.A, env.B, C, D, 
    A_dependencies=env.A_dependencies, 
    B_dependencies=env.B_dependencies,
    policy_len=2,
    learn_A=False,
    learn_B=False,
    action_selection="deterministic",
    sampling_mode="full",
    gamma=3.0
)

agent_si = Agent(
    env.A, env.B, C, D, 
    A_dependencies=env.A_dependencies, 
    B_dependencies=env.B_dependencies,
    policy_len=1,
    learn_A=False,
    learn_B=False,
    action_selection="deterministic",
    sampling_mode="full",
    gamma=3.0
)

### Running the active inference rollouts

In [7]:
key = jr.PRNGKey(0) 
T = 3

In [8]:
si_search = si_policy_search(
    horizon=2, # plans 2 timesteps ahead
    max_nodes=5000, # maximum number of nodes allowed in the tree
    max_branching=10, # maximum number of children allowed per node (moderating the branching factor)
    policy_prune_threshold=0.0, # no pruning of unlikely policies
    observation_prune_threshold=0.0, # no pruning of unlikely observations
    entropy_stop_threshold=0.0, # disabling halting of expansion if agent is certain enough
    efe_stop_threshold=1e10, # disabling efe value based halting of expansion
    kl_threshold=-1, # disabling node reuse if agent is in similar states after an action
    prune_penalty=512, # default value for prune penalty
    gamma=3, # temperature parameter; lower value (--> 1) prunes policies less aggressively as probabilities are flattened while higher value (--> 16) prunes more aggressively
    topk_obsspace=10000, # max number of top observation combinations - this default value just means we want to consider all the observation combinations
)

In [9]:
_, info_vanilla = rollout(agent_vanilla, env, num_timesteps=T, rng_key=key) # default policy search is vanilla
_, info_si = rollout(agent_si, env, num_timesteps=T, rng_key=key, policy_search=si_search)

/Users/riddhi/Documents/Projects/pymdp/.venv/lib/python3.11/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float32 to dtype=int32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
/Users/riddhi/Documents/Projects/pymdp/.venv/lib/python3.11/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [10]:
def make_gif(info):
    frames = []
    for t in range(info["observation"][0].shape[1]):  # iterate over timesteps
        # get observations for this timestep
        observations_t = [
            info["observation"][0][:, t, :],
            info["observation"][1][:, t, :],  
        ]
        
        frame = env.render(mode="rgb_array", observations=observations_t) # render the environment using the observations for this timestep
        frame = np.asarray(frame, dtype=np.uint8)
        plt.close()  # close the figure to prevent memory leak
        frames.append(frame)

    frames = np.array(frames, dtype=np.uint8)
    mediapy.show_video(frames, fps=1)

In [11]:
# make_gif(info_vanilla)

In [12]:
# make_gif(info_si)

### Result analysis

In [13]:
# helper functions for:
# - printing out policies and respective probabilities of selecting those policies
# - printing out action and observation info for each timestep

np.set_printoptions(precision=2, suppress=True)

def print_qpi(agent, info, print_t1=False):
    policies = agent.policies
    qpi_values = info["qpi"]
    
    action_names = {0: "move to centre", 1: "move to left arm", 2: "move to right arm", 3: "move to cue"}
    max_timesteps = 1 if print_t1 else qpi_values.shape[1]
    
    for t in range(max_timesteps):
        print(f"Timestep {t}:")
        
        # grouping policies by their first action
        first_action_groups = {}
        
        for i in range(len(policies)):
            first_action = tuple(policies[i][0].tolist())  
            qpi_val = qpi_values[0, t, i]
            
            if first_action not in first_action_groups:
                first_action_groups[first_action] = 0.0
            
            first_action_groups[first_action] += qpi_val
        
        # print the summed probabilities for each first action with readable names
        for first_action, total_prob in sorted(first_action_groups.items()):
            action_name = action_names.get(first_action[0], f"action_{first_action[0]}")
            print(f"  {action_name}: {total_prob:.3f}")
        print()

def print_agent_behaviour(info):

    action_names = {0: "move to centre", 1: "move to left arm", 2: "move to right arm", 3: "move to cue"}
    
    location_obs = {0: "centre loc", 1: "left arm loc", 2: "right arm loc", 3: "cue-left-arm", 4: "cue-right-arm"}
    outcome_obs = {0: "no_outcome", 1: "reward", 2: "punishment"}
    
    actions = info["action"]
    observations = info["observation"]
    
    num_timesteps = actions.shape[1]
    
    for t in range(num_timesteps):
        action_idx = int(actions[0, t, 0])  # [batch, timestep, action_dim]
        action_name = action_names.get(action_idx, f"action_{action_idx}")
        
        location_obs_idx = int(observations[0][0, t, 0])  # [modality][batch, timestep, obs_dim]
        outcome_obs_idx = int(observations[1][0, t, 0])
        
        location_name = location_obs.get(location_obs_idx)
        outcome_name = outcome_obs.get(outcome_obs_idx)
        
        print(f"t={t}: observed=({location_name}, {outcome_name}) -> action={action_name}")

We can see both agents select the optimal actions to go to the cue first and then go to the left arm to get a reward. 

In [14]:
print_agent_behaviour(info_vanilla)

t=0: observed=(centre loc, no_outcome) -> action=move to cue
t=1: observed=(cue-left-arm, no_outcome) -> action=move to left arm
t=2: observed=(left arm loc, reward) -> action=move to left arm
t=3: observed=(left arm loc, reward) -> action=move to left arm


In [15]:
print_agent_behaviour(info_si)

t=0: observed=(centre loc, no_outcome) -> action=move to cue
t=1: observed=(cue-left-arm, no_outcome) -> action=move to left arm
t=2: observed=(left arm loc, reward) -> action=move to left arm
t=3: observed=(left arm loc, reward) -> action=move to left arm


Now, to see how the two planning algorithms differ, let's examine how they evaluate policies... 

While both agents select the optimal strategy (information gathering first), they differ significantly in their confidence, with the SI agent shows much stronger preference for the information-gathering strategy.
- Vanilla Agent: 81% probability for cue-seeking, 18% for staying at centre
- SI Agent: 98% probability for cue-seeking, <1% for staying at centre

In [16]:
print_qpi(agent_vanilla, info_vanilla, print_t1=True)

Timestep 0:
  move to centre: 0.183
  move to left arm: 0.004
  move to right arm: 0.004
  move to cue: 0.809



In [17]:
print_qpi(agent_si, info_si, print_t1=True)

Timestep 0:
  move to centre: 0.003
  move to left arm: 0.008
  move to right arm: 0.008
  move to cue: 0.980

